<a href="https://colab.research.google.com/github/JaquelineLopezCh/actividad4-huggingface-nlp/blob/main/Actividad4ModelosHuggingFaceNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Actividad 4
# Implementación y comparación de modelos preentrenados
# Hugging Face + Google Colab + GPU
# Rama seleccionada: Procesamiento de Lenguaje Natural
# Caso de uso: Clasificación automática de sentimiento

# 1. Instalación de dependencias


!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy openpyxl huggingface_hub

# 2. Importación de librerías


import os
import time
import json
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from huggingface_hub import notebook_login


# 3. Configuración del entorno de ejecución


print("==============================================")
print("CONFIGURACIÓN DEL ENTORNO")
print("==============================================")
print("Versión de PyTorch:", torch.__version__)
print("¿GPU disponible?:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = 0
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    device = -1
    print("No se detectó GPU. El notebook se ejecutará en CPU.")

print("Dispositivo utilizado por pipeline:", "GPU" if device == 0 else "CPU")



# 4. Autenticación en Hugging Face Hub


print("\n==============================================")
print("AUTENTICACIÓN EN HUGGING FACE HUB")
print("==============================================")

# La actividad solicita autenticación en Hugging Face Hub.
# Si ya hiciste login antes, esta parte puede reconocer tu sesión.
# Si no quieres autenticarte de nuevo, cambia AUTHENTICATE = True por AUTHENTICATE = False.

AUTHENTICATE = True

if AUTHENTICATE:
    try:
        notebook_login()
    except Exception as e:
        print("No se completó la autenticación.")
        print("El notebook continuará porque los modelos y datasets utilizados son públicos.")
        print("Detalle:", e)
else:
    print("Autenticación omitida para esta ejecución.")


# 5. Definición del caso de uso


case_context = {
    "rama_ia": "Procesamiento de Lenguaje Natural",
    "caso_uso": "Clasificación automática de sentimiento en textos",
    "objetivo": "Comparar modelos preentrenados de Hugging Face para identificar el modelo más adecuado para clasificar textos como positivos o negativos.",
    "aplicacion_empresarial": "Este tipo de prototipo puede extenderse a análisis de comentarios de usuarios, encuestas internas, correos, tickets de soporte o mensajes operativos en procesos empresariales."
}

print("\n==============================================")
print("CASO DE USO")
print("==============================================")
for key, value in case_context.items():
    print(f"{key}: {value}")



# 6. Carga del dataset GLUE SST-2


print("\n==============================================")
print("CARGA DEL DATASET")
print("==============================================")

# IMPORTANTE:
# Se usa "nyu-mll/glue" en lugar de "glue" para evitar el error:
# Repository id must be 'namespace/name', got 'glue'.

dataset = load_dataset("nyu-mll/glue", "sst2")

print(dataset)
print("\nEjemplo del dataset de validación:")
print(dataset["validation"][0])


# 7. Preparación del dataset


# Se utiliza el conjunto de validación porque contiene etiquetas reales.
validation_data = dataset["validation"]

df = pd.DataFrame(validation_data)

# Renombrar columnas para mayor claridad.
df = df.rename(columns={
    "sentence": "text",
    "label": "true_label"
})

# Tamaño de muestra para evaluación.
# Puedes subirlo a 300 o 500 si el tiempo de ejecución es aceptable.
sample_size = 200

df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

df_sample["true_label_text"] = df_sample["true_label"].map({
    0: "NEGATIVE",
    1: "POSITIVE"
})

print("\n==============================================")
print("MUESTRA DEL DATASET")
print("==============================================")
print("Tamaño total del dataset de validación:", len(df))
print("Tamaño de muestra utilizado:", len(df_sample))
display(df_sample.head())

# 8. Crear estructura de carpetas para el entregable


folders = [
    "data",
    "results",
    "docs",
    "src",
    "notebooks"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("\nCarpetas creadas correctamente.")

# 9. Guardar dataset utilizado

df_sample.to_csv("data/sst2_sample_dataset.csv", index=False, encoding="utf-8")

print("\nArchivo generado:")
print("data/sst2_sample_dataset.csv")


# 10. Definición de modelos preentrenados a evaluar


models_to_evaluate = [
    {
        "model_name": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
        "short_name": "DistilBERT SST-2",
        "task": "sentiment-analysis",
        "architecture": "DistilBERT",
        "size_category": "Ligero",
        "license": "Apache-2.0",
        "selection_reason": "Modelo ligero, entrenado específicamente para SST-2 y adecuado para prototipos rápidos con menor consumo de recursos."
    },
    {
        "model_name": "textattack/bert-base-uncased-SST-2",
        "short_name": "BERT SST-2",
        "task": "sentiment-analysis",
        "architecture": "BERT",
        "size_category": "Medio",
        "license": "No especificada claramente en la tarjeta del modelo",
        "selection_reason": "Modelo basado en BERT para clasificación de sentimiento en SST-2, útil como punto de comparación frente a arquitecturas ligeras."
    },
    {
        "model_name": "siebert/sentiment-roberta-large-english",
        "short_name": "RoBERTa Large Sentiment",
        "task": "sentiment-analysis",
        "architecture": "RoBERTa Large",
        "size_category": "Grande",
        "license": "No especificada claramente en la tarjeta del modelo",
        "selection_reason": "Modelo robusto para análisis de sentimiento en inglés, entrenado para generalizar en diferentes tipos de texto."
    }
]

models_df = pd.DataFrame(models_to_evaluate)

print("\n==============================================")
print("MODELOS A EVALUAR")
print("==============================================")
display(models_df)

# 11. Función para normalizar etiquetas

def normalize_prediction_label(label):
    """
    Convierte diferentes formatos de salida de modelos de Hugging Face
    a etiquetas binarias compatibles con SST-2.

    0 = NEGATIVE
    1 = POSITIVE
    """
    label = str(label).upper().strip()

    if label in ["POSITIVE", "LABEL_1", "1"]:
        return 1
    elif label in ["NEGATIVE", "LABEL_0", "0"]:
        return 0
    else:
        return np.nan

# 12. Función para evaluar modelos

def evaluate_model(model_info, dataframe, device):
    """
    Carga un modelo preentrenado de Hugging Face, ejecuta inferencia,
    calcula métricas estándar y mide latencia.

    Métricas:
    - Accuracy
    - Precision
    - Recall
    - F1-score
    - Latencia promedio
    """

    model_name = model_info["model_name"]
    short_name = model_info["short_name"]

    print("\n" + "=" * 90)
    print(f"EVALUANDO MODELO: {short_name}")
    print(f"Modelo Hugging Face: {model_name}")
    print("=" * 90)

    classifier = pipeline(
        task=model_info["task"],
        model=model_name,
        tokenizer=model_name,
        device=device
    )

    texts = dataframe["text"].tolist()
    y_true = dataframe["true_label"].tolist()

    predictions = []
    scores = []
    latencies = []
    raw_labels = []

    start_total_time = time.time()

    for text in texts:
        start_time = time.time()

        result = classifier(
            text,
            truncation=True,
            max_length=512
        )[0]

        end_time = time.time()

        raw_label = result["label"]
        pred_label = normalize_prediction_label(raw_label)

        raw_labels.append(raw_label)
        predictions.append(pred_label)
        scores.append(result["score"])
        latencies.append(end_time - start_time)

    end_total_time = time.time()

    result_df = dataframe.copy()
    result_df[f"{short_name}_raw_label"] = raw_labels
    result_df[f"{short_name}_prediction"] = predictions
    result_df[f"{short_name}_score"] = scores
    result_df[f"{short_name}_latency_seconds"] = latencies

    valid_mask = ~pd.isna(pd.Series(predictions))

    y_true_valid = pd.Series(y_true)[valid_mask].astype(int).tolist()
    y_pred_valid = pd.Series(predictions)[valid_mask].astype(int).tolist()

    accuracy = accuracy_score(y_true_valid, y_pred_valid)
    precision = precision_score(y_true_valid, y_pred_valid, average="binary", zero_division=0)
    recall = recall_score(y_true_valid, y_pred_valid, average="binary", zero_division=0)
    f1 = f1_score(y_true_valid, y_pred_valid, average="binary", zero_division=0)

    avg_latency = float(np.mean(latencies))
    total_inference_time = float(end_total_time - start_total_time)

    summary = {
        "model": short_name,
        "huggingface_model": model_name,
        "task": model_info["task"],
        "architecture": model_info["architecture"],
        "size_category": model_info["size_category"],
        "license": model_info["license"],
        "sample_size": len(dataframe),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "avg_latency_seconds": avg_latency,
        "total_inference_time_seconds": total_inference_time,
        "selection_reason": model_info["selection_reason"]
    }

    print("\nResumen de métricas:")
    for key, value in summary.items():
        print(f"{key}: {value}")

    print("\nReporte de clasificación:")
    print(classification_report(
        y_true_valid,
        y_pred_valid,
        target_names=["NEGATIVE", "POSITIVE"],
        zero_division=0
    ))

    return summary, result_df

# 13. Evaluación de los tres modelos


all_summaries = []
all_prediction_results = {}

for model_info in models_to_evaluate:
    try:
        summary, result_df = evaluate_model(model_info, df_sample, device)
        all_summaries.append(summary)
        all_prediction_results[model_info["short_name"]] = result_df

    except Exception as e:
        print("\nERROR evaluando el modelo:", model_info["short_name"])
        print("Detalle del error:", e)
        print("El proceso continuará con los demás modelos.")

# 14. Tabla comparativa de resultados

comparison_df = pd.DataFrame(all_summaries)

if len(comparison_df) == 0:
    raise ValueError("No se pudo evaluar ningún modelo. Revisa los errores anteriores.")

comparison_df = comparison_df.sort_values(
    by=["f1_score", "avg_latency_seconds"],
    ascending=[False, True]
).reset_index(drop=True)

print("\n==============================================")
print("TABLA COMPARATIVA FINAL")
print("==============================================")
display(comparison_df)


# 15. Guardar resultados comparativos


comparison_df.to_csv("results/model_comparison_results.csv", index=False, encoding="utf-8")
comparison_df.to_excel("results/model_comparison_summary.xlsx", index=False)

print("\nArchivos generados:")
print("results/model_comparison_results.csv")
print("results/model_comparison_summary.xlsx")


# 16. Consolidar predicciones detalladas

detailed_results = df_sample.copy()

for model_short_name, result_df in all_prediction_results.items():
    detailed_results[f"{model_short_name}_raw_label"] = result_df[f"{model_short_name}_raw_label"]
    detailed_results[f"{model_short_name}_prediction"] = result_df[f"{model_short_name}_prediction"]
    detailed_results[f"{model_short_name}_score"] = result_df[f"{model_short_name}_score"]
    detailed_results[f"{model_short_name}_latency_seconds"] = result_df[f"{model_short_name}_latency_seconds"]

detailed_results.to_csv("results/detailed_model_predictions.csv", index=False, encoding="utf-8")
detailed_results.to_excel("results/detailed_model_predictions.xlsx", index=False)

print("\nArchivos generados:")
print("results/detailed_model_predictions.csv")
print("results/detailed_model_predictions.xlsx")

display(detailed_results.head())

# 17. Selección automática del modelo recomendado


best_model = comparison_df.iloc[0]

print("\n==============================================")
print("MODELO RECOMENDADO")
print("==============================================")

print("Modelo recomendado:", best_model["model"])
print("Modelo en Hugging Face:", best_model["huggingface_model"])
print("F1-score:", round(best_model["f1_score"], 4))
print("Accuracy:", round(best_model["accuracy"], 4))
print("Precision:", round(best_model["precision"], 4))
print("Recall:", round(best_model["recall"], 4))
print("Latencia promedio:", round(best_model["avg_latency_seconds"], 4), "segundos")

# 18. Generación de conclusiones académicas

conclusion_text = f"""
Conclusiones académicas del experimento

La presente actividad permitió implementar y comparar modelos preentrenados de Hugging Face en una tarea de Procesamiento de Lenguaje Natural orientada a la clasificación automática de sentimiento. El caso de uso se centró en clasificar textos como positivos o negativos, utilizando el dataset SST-2 como base de evaluación.

Los resultados obtenidos muestran que la selección de un modelo no debe depender únicamente de la exactitud alcanzada. Aunque métricas como accuracy, precision, recall y F1-score permiten evaluar la calidad predictiva, la latencia promedio también representa un criterio relevante para determinar la viabilidad operativa del modelo en un contexto real.

En la evaluación realizada, el modelo con mejor balance de desempeño fue {best_model['model']}. Este modelo obtuvo un F1-score de {best_model['f1_score']:.4f}, accuracy de {best_model['accuracy']:.4f}, precision de {best_model['precision']:.4f}, recall de {best_model['recall']:.4f} y una latencia promedio de {best_model['avg_latency_seconds']:.4f} segundos por texto.

Desde una perspectiva práctica, este resultado permite concluir que el modelo recomendado ofrece una relación adecuada entre precisión y eficiencia. Esto es especialmente importante en escenarios donde se requiere procesar un volumen considerable de textos, como comentarios de usuarios, tickets de soporte, encuestas internas o comunicaciones operativas.

El uso de Google Colab con soporte GPU facilitó el prototipado rápido y reproducible, mientras que el ecosistema Hugging Face permitió integrar modelos, datasets y pipelines de inferencia de manera estructurada. En conjunto, el ejercicio demuestra la importancia de realizar una evaluación comparativa antes de seleccionar un modelo preentrenado para una posible implementación empresarial.
"""

print("\n==============================================")
print("CONCLUSIONES ACADÉMICAS")
print("==============================================")
print(conclusion_text)

with open("docs/conclusiones_academicas.txt", "w", encoding="utf-8") as file:
    file.write(conclusion_text)

print("\nArchivo generado:")
print("docs/conclusiones_academicas.txt")


# 19. Descripción del dataset para el entregable

dataset_description = """
Descripción del dataset

Nombre del dataset:
GLUE SST-2

Fuente:
Hugging Face Datasets, mediante la librería datasets.

Procedimiento de carga:
El dataset fue cargado con la función load_dataset("nyu-mll/glue", "sst2").

Estructura de los datos:
- text: frase o texto que será clasificado.
- true_label: etiqueta numérica original del dataset.
  - 0: sentimiento negativo.
  - 1: sentimiento positivo.
- true_label_text: etiqueta textual generada para facilitar la interpretación.
- idx: identificador original del registro en el dataset.

Uso en el prototipo:
Se utilizó una muestra aleatoria del conjunto de validación para evaluar el desempeño de tres modelos preentrenados de Hugging Face. El conjunto de validación fue seleccionado porque contiene etiquetas reales, lo cual permite calcular métricas como accuracy, precision, recall y F1-score.

Consideraciones:
El dataset se encuentra en inglés, por lo que los modelos seleccionados también corresponden a modelos entrenados o ajustados para clasificación de sentimiento en inglés.
"""

with open("docs/dataset_description.txt", "w", encoding="utf-8") as file:
    file.write(dataset_description)

print("\nArchivo generado:")
print("docs/dataset_description.txt")

# 20. Generar README.md para GitHub

readme_content = f"""
# Actividad 4: Comparación de modelos preentrenados de Hugging Face

## Descripción del proyecto

Este repositorio contiene el prototipo desarrollado para la Actividad 4 de la materia Gestión de Proyectos de Inteligencia Artificial. El objetivo del proyecto es implementar, evaluar y comparar modelos preentrenados de Hugging Face para un caso de uso de Procesamiento de Lenguaje Natural.

## Rama de inteligencia artificial seleccionada

Procesamiento de Lenguaje Natural, NLP.

## Caso de uso

Clasificación automática de sentimiento en textos. El prototipo clasifica frases como positivas o negativas utilizando modelos preentrenados disponibles en Hugging Face.

## Dataset

Se utilizó el dataset GLUE SST-2, cargado mediante la librería datasets de Hugging Face.

Procedimiento de carga:

load_dataset("nyu-mll/glue", "sst2")

Campos principales:

- text: texto o frase a clasificar.
- true_label: etiqueta numérica original.
- true_label_text: etiqueta textual.
- idx: identificador original del registro.

## Modelos evaluados

1. distilbert/distilbert-base-uncased-finetuned-sst-2-english
2. textattack/bert-base-uncased-SST-2
3. siebert/sentiment-roberta-large-english

## Métricas utilizadas

- Accuracy
- Precision
- Recall
- F1-score
- Latencia promedio por inferencia

## Entorno de ejecución

El prototipo fue desarrollado en Google Colab utilizando Python y, cuando estuvo disponible, aceleración por GPU.

## Dependencias principales

- transformers
- datasets
- evaluate
- accelerate
- scikit-learn
- pandas
- numpy
- openpyxl
- huggingface_hub
- torch

## Resultados

Los resultados comparativos se encuentran en la carpeta results.

Archivos principales:

- results/model_comparison_results.csv
- results/model_comparison_summary.xlsx
- results/detailed_model_predictions.csv
- results/detailed_model_predictions.xlsx

## Modelo recomendado

El modelo recomendado de acuerdo con los resultados obtenidos fue:

**{best_model['model']}**

Este modelo obtuvo:

- Accuracy: {best_model['accuracy']:.4f}
- Precision: {best_model['precision']:.4f}
- Recall: {best_model['recall']:.4f}
- F1-score: {best_model['f1_score']:.4f}
- Latencia promedio: {best_model['avg_latency_seconds']:.4f} segundos

## Conclusión

La evaluación permitió identificar que la selección de un modelo preentrenado debe considerar tanto el desempeño predictivo como la eficiencia computacional. El F1-score permitió comparar el balance entre precision y recall, mientras que la latencia permitió analizar la viabilidad del modelo para una posible implementación práctica.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("\nArchivo generado:")
print("README.md")


# 21. Generar requirements.txt


requirements_content = """
transformers
datasets
evaluate
accelerate
scikit-learn
pandas
numpy
openpyxl
huggingface_hub
torch
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements_content.strip())

print("\nArchivo generado:")
print("requirements.txt")

# 22. Generar script de soporte src/evaluate_models.py

support_script = '''
"""
Script de soporte para la Actividad 4.
Este archivo documenta de forma reproducible la lógica principal utilizada
para evaluar modelos preentrenados de Hugging Face en una tarea de clasificación
de sentimiento.

Nota:
El notebook de Google Colab contiene la ejecución completa del prototipo.
"""

import time
import numpy as np
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def normalize_prediction_label(label):
    label = str(label).upper().strip()

    if label in ["POSITIVE", "LABEL_1", "1"]:
        return 1
    elif label in ["NEGATIVE", "LABEL_0", "0"]:
        return 0
    else:
        return np.nan


def evaluate_model(model_info, dataframe, device=-1):
    model_name = model_info["model_name"]
    short_name = model_info["short_name"]

    classifier = pipeline(
        task=model_info["task"],
        model=model_name,
        tokenizer=model_name,
        device=device
    )

    texts = dataframe["text"].tolist()
    y_true = dataframe["true_label"].tolist()

    predictions = []
    latencies = []

    start_total_time = time.time()

    for text in texts:
        start_time = time.time()

        result = classifier(
            text,
            truncation=True,
            max_length=512
        )[0]

        end_time = time.time()

        predictions.append(normalize_prediction_label(result["label"]))
        latencies.append(end_time - start_time)

    end_total_time = time.time()

    valid_mask = ~pd.isna(pd.Series(predictions))

    y_true_valid = pd.Series(y_true)[valid_mask].astype(int).tolist()
    y_pred_valid = pd.Series(predictions)[valid_mask].astype(int).tolist()

    summary = {
        "model": short_name,
        "huggingface_model": model_name,
        "accuracy": accuracy_score(y_true_valid, y_pred_valid),
        "precision": precision_score(y_true_valid, y_pred_valid, average="binary", zero_division=0),
        "recall": recall_score(y_true_valid, y_pred_valid, average="binary", zero_division=0),
        "f1_score": f1_score(y_true_valid, y_pred_valid, average="binary", zero_division=0),
        "avg_latency_seconds": float(np.mean(latencies)),
        "total_inference_time_seconds": float(end_total_time - start_total_time)
    }

    return summary
'''

with open("src/evaluate_models.py", "w", encoding="utf-8") as file:
    file.write(support_script)

print("\nArchivo generado:")
print("src/evaluate_models.py")


# 23. Crear archivo JSON con configuración del experimento


experiment_config = {
    "project": "Actividad 4 - Hugging Face NLP",
    "ai_branch": "Procesamiento de Lenguaje Natural",
    "use_case": "Clasificación automática de sentimiento",
    "dataset": "GLUE SST-2",
    "dataset_loading": 'load_dataset("nyu-mll/glue", "sst2")',
    "sample_size": sample_size,
    "models": models_to_evaluate,
    "recommended_model": {
        "model": best_model["model"],
        "huggingface_model": best_model["huggingface_model"],
        "accuracy": float(best_model["accuracy"]),
        "precision": float(best_model["precision"]),
        "recall": float(best_model["recall"]),
        "f1_score": float(best_model["f1_score"]),
        "avg_latency_seconds": float(best_model["avg_latency_seconds"])
    }
}

with open("results/experiment_config.json", "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4, ensure_ascii=False)

print("\nArchivo generado:")
print("results/experiment_config.json")


# 24. Crear archivo ZIP del entregable

!zip -r actividad_4_huggingface_nlp.zip README.md requirements.txt data results docs src

print("\n==============================================")
print("PROCESO FINALIZADO")
print("==============================================")
print("Archivo ZIP generado:")
print("actividad_4_huggingface_nlp.zip")

print("\nArchivos principales que debes descargar:")
print("- actividad_4_huggingface_nlp.zip")
print("- Tu notebook de Colab en formato .ipynb")
print("- data/sst2_sample_dataset.csv")
print("- results/model_comparison_summary.xlsx")
print("- results/model_comparison_results.csv")

CONFIGURACIÓN DEL ENTORNO
Versión de PyTorch: 2.11.0+cu128
¿GPU disponible?: True
GPU detectada: Tesla T4
Dispositivo utilizado por pipeline: GPU

AUTENTICACIÓN EN HUGGING FACE HUB

CASO DE USO
rama_ia: Procesamiento de Lenguaje Natural
caso_uso: Clasificación automática de sentimiento en textos
objetivo: Comparar modelos preentrenados de Hugging Face para identificar el modelo más adecuado para clasificar textos como positivos o negativos.
aplicacion_empresarial: Este tipo de prototipo puede extenderse a análisis de comentarios de usuarios, encuestas internas, correos, tickets de soporte o mensajes operativos en procesos empresariales.

CARGA DEL DATASET
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

Ejemplo del datas

,text,true_label,idx,true_label_text
0,it confirms fincher 's status as a film maker ...,1,794,POSITIVE
1,too much of it feels unfocused and underdevelo...,0,319,NEGATIVE
2,a great ensemble cast ca n't lift this heartfe...,0,382,NEGATIVE
3,"prurient playthings aside , there 's little to...",0,779,NEGATIVE
4,"it moves quickly , adroitly , and without fuss...",1,422,POSITIVE



Carpetas creadas correctamente.

Archivo generado:
data/sst2_sample_dataset.csv

MODELOS A EVALUAR


,model_name,short_name,task,architecture,size_category,license,selection_reason
0,distilbert/distilbert-base-uncased-finetuned-s...,DistilBERT SST-2,sentiment-analysis,DistilBERT,Ligero,Apache-2.0,"Modelo ligero, entrenado específicamente para ..."
1,textattack/bert-base-uncased-SST-2,BERT SST-2,sentiment-analysis,BERT,Medio,No especificada claramente en la tarjeta del m...,Modelo basado en BERT para clasificación de se...
2,siebert/sentiment-roberta-large-english,RoBERTa Large Sentiment,sentiment-analysis,RoBERTa Large,Grande,No especificada claramente en la tarjeta del m...,Modelo robusto para análisis de sentimiento en...



EVALUANDO MODELO: DistilBERT SST-2
Modelo Hugging Face: distilbert/distilbert-base-uncased-finetuned-sst-2-english


...


Resumen de métricas:
model: RoBERTa Large Sentiment
huggingface_model: siebert/sentiment-roberta-large-english
task: sentiment-analysis
architecture: RoBERTa Large
size_category: Grande
license: No especificada claramente en la tarjeta del modelo
sample_size: 200
accuracy: 0.945
precision: 0.9375
recall: 0.963302752293578
f1_score: 0.9502262443438914
avg_latency_seconds: 0.016512396335601805
total_inference_time_seconds: 3.303220748901367
selection_reason: Modelo robusto para análisis de sentimiento en inglés, entrenado para generalizar en diferentes tipos de texto.

Reporte de clasificación:
              precision    recall  f1-score   support

    NEGATIVE       0.95      0.92      0.94        91
    POSITIVE       0.94      0.96      0.95       109

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200


TABLA COMPARATIVA FINAL


,model,huggingface_model,task,architecture,size_category,license,sample_size,accuracy,precision,recall,f1_score,avg_latency_seconds,total_inference_time_seconds,selection_reason
0,RoBERTa Large Sentiment,siebert/sentiment-roberta-large-english,sentiment-analysis,RoBERTa Large,Grande,No especificada claramente en la tarjeta del m...,200,0.945,0.937500,0.963303,0.950226,0.016512,3.303221,Modelo robusto para análisis de sentimiento en...
1,BERT SST-2,textattack/bert-base-uncased-SST-2,sentiment-analysis,BERT,Medio,No especificada claramente en la tarjeta del m...,200,0.945,0.953704,0.944954,0.949309,0.010962,2.193153,Modelo basado en BERT para clasificación de se...
2,DistilBERT SST-2,distilbert/distilbert-base-uncased-finetuned-s...,sentiment-analysis,DistilBERT,Ligero,Apache-2.0,200,0.935,0.928571,0.954128,0.941176,0.005023,1.005306,"Modelo ligero, entrenado específicamente para ..."



Archivos generados:
results/model_comparison_results.csv
results/model_comparison_summary.xlsx

Archivos generados:
results/detailed_model_predictions.csv
results/detailed_model_predictions.xlsx


,text,true_label,idx,true_label_text,DistilBERT SST-2_raw_label,DistilBERT SST-2_prediction,DistilBERT SST-2_score,DistilBERT SST-2_latency_seconds,BERT SST-2_raw_label,BERT SST-2_prediction,BERT SST-2_score,BERT SST-2_latency_seconds,RoBERTa Large Sentiment_raw_label,RoBERTa Large Sentiment_prediction,RoBERTa Large Sentiment_score,RoBERTa Large Sentiment_latency_seconds
0,it confirms fincher 's status as a film maker ...,1,794,POSITIVE,POSITIVE,1,0.999364,0.006371,LABEL_1,1,0.999491,0.011716,POSITIVE,1,0.998918,0.018758
1,too much of it feels unfocused and underdevelo...,0,319,NEGATIVE,NEGATIVE,0,0.999767,0.005231,LABEL_0,0,0.998887,0.011269,NEGATIVE,0,0.999505,0.018677
2,a great ensemble cast ca n't lift this heartfe...,0,382,NEGATIVE,NEGATIVE,0,0.996946,0.005125,LABEL_0,0,0.812849,0.013010,NEGATIVE,0,0.998354,0.015924
3,"prurient playthings aside , there 's little to...",0,779,NEGATIVE,NEGATIVE,0,0.999493,0.005053,LABEL_0,0,0.998263,0.010970,NEGATIVE,0,0.999505,0.015831
4,"it moves quickly , adroitly , and without fuss...",1,422,POSITIVE,POSITIVE,1,0.645543,0.005887,LABEL_0,0,0.864377,0.011714,POSITIVE,1,0.998520,0.022167



MODELO RECOMENDADO
Modelo recomendado: RoBERTa Large Sentiment
Modelo en Hugging Face: siebert/sentiment-roberta-large-english
F1-score: 0.9502
Accuracy: 0.945
Precision: 0.9375
Recall: 0.9633
Latencia promedio: 0.0165 segundos

CONCLUSIONES ACADÉMICAS

Conclusiones académicas del experimento

La presente actividad permitió implementar y comparar modelos preentrenados de Hugging Face en una tarea de Procesamiento de Lenguaje Natural orientada a la clasificación automática de sentimiento. El caso de uso se centró en clasificar textos como positivos o negativos, utilizando el dataset SST-2 como base de evaluación.

Los resultados obtenidos muestran que la selección de un modelo no debe depender únicamente de la exactitud alcanzada. Aunque métricas como accuracy, precision, recall y F1-score permiten evaluar la calidad predictiva, la latencia promedio también representa un criterio relevante para determinar la viabilidad operativa del modelo en un contexto real.

En la evaluación realiza